# Itchy — Finding the Optimal Model

Test larger patch sizes + compound with research findings.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

Expected time: ~40 min

In [ ]:
!pip install -q torch numpy huggingface-hub sentencepiece
import torch, math, time, os, shutil, numpy as np
import torch.nn.functional as F
from torch import nn
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
print(f'GPU: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda')

In [ ]:
# Data
os.makedirs('data/bytes', exist_ok=True)
def dl(fn,sub,dst):
    if Path(dst).exists(): return
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn,subfolder=sub,repo_type='dataset')).resolve())
    try: os.link(c,dst)
    except: shutil.copy2(c,dst)
dl('fineweb_train_000000.bin','datasets/datasets/fineweb10B_sp1024','data/train.bin')
dl('fineweb_val_000000.bin','datasets/datasets/fineweb10B_sp1024','data/val.bin')
dl('fineweb_1024_bpe.model','datasets/tokenizers','data/tok.model')
def load_shard(p):
    h=np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4)
def write_shard(p,d):
    h=np.zeros(256,dtype='<i4');h[0]=20240520;h[1]=1;h[2]=len(d)
    with open(p,'wb') as f: f.write(h.tobytes());f.write(d.astype('<u2').tobytes())
if not Path('data/bytes/train.bin').exists():
    sp=spm.SentencePieceProcessor(model_file='data/tok.model')
    for s,d in [('data/train.bin','data/bytes/train.bin'),('data/val.bin','data/bytes/val.bin')]:
        print(f'Converting {s}...')
        tok=load_shard(s)
        bs=[np.frombuffer(sp.decode(tok[i:i+100000].tolist()).encode('utf-8'),dtype=np.uint8)
            for i in range(0,len(tok),100000)]
        write_shard(d,np.concatenate(bs))
def load_bytes(p):
    h=np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4).astype(np.int64)
train_data=load_bytes('data/bytes/train.bin')
val_data=load_bytes('data/bytes/val.bin')
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# ============================================================
# MODEL — supports all features toggled on/off
# ============================================================

class Rotary(nn.Module):
    def __init__(self,dim,base=10000.0):
        super().__init__()
        inv_freq=1.0/(base**(torch.arange(0,dim,2,dtype=torch.float32)/dim))
        self.register_buffer('inv_freq',inv_freq,persistent=False)
        self._cache=None
    def forward(self,seq_len,device,dtype):
        if self._cache is None or self._cache[0]!=seq_len:
            t=torch.arange(seq_len,device=device,dtype=self.inv_freq.dtype)
            freqs=torch.outer(t,self.inv_freq.to(device))
            self._cache=(seq_len,freqs.cos()[None,None,:,:],freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype),self._cache[2].to(dtype)

def apply_rope(x,cos,sin):
    h=x.size(-1)//2
    x1,x2=x[...,:h],x[...,h:]
    return torch.cat((x1*cos+x2*sin,x1*(-sin)+x2*cos),dim=-1)

class Attn(nn.Module):
    def __init__(self,dim,nh,nkv,qk_gain):
        super().__init__()
        self.nh,self.nkv,self.hd=nh,nkv,dim//nh
        kv=nkv*self.hd
        self.cq=nn.Linear(dim,dim,bias=False)
        self.ck=nn.Linear(dim,kv,bias=False)
        self.cv=nn.Linear(dim,kv,bias=False)
        self.proj=nn.Linear(dim,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg=nn.Parameter(torch.full((nh,),qk_gain))
        self.rot=Rotary(self.hd)
    def forward(self,x):
        B,S,D=x.shape
        q=self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k=self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v=self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q,k=F.rms_norm(q,(q.size(-1),)),F.rms_norm(k,(k.size(-1),))
        cos,sin=self.rot(S,x.device,q.dtype)
        q,k=apply_rope(q,cos,sin),apply_rope(k,cos,sin)
        q=q*self.qg.to(q.dtype)[None,:,None,None]
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True,enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self,dim,mult):
        super().__init__()
        self.fc=nn.Linear(dim,dim*mult,bias=False)
        self.proj=nn.Linear(dim*mult,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self,x):
        return self.proj(F.leaky_relu(self.fc(x),0.5).square())

class Block(nn.Module):
    def __init__(self,dim,nh,nkv,mlp_mult,qk_gain):
        super().__init__()
        self.attn=Attn(dim,nh,nkv,qk_gain)
        self.mlp=MLP(dim,mlp_mult)
        self.as_=nn.Parameter(torch.ones(dim))
        self.ms=nn.Parameter(torch.ones(dim))
        self.rm=nn.Parameter(torch.stack((torch.ones(dim),torch.zeros(dim))))
    def forward(self,x,x0):
        m=self.rm.to(x.dtype)
        x=m[0][None,None,:]*x+m[1][None,None,:]*x0
        x=x+self.as_.to(x.dtype)[None,None,:]*self.attn(F.rms_norm(x,(x.size(-1),)))
        x=x+self.ms.to(x.dtype)[None,None,:]*self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x

class LocalEncoder(nn.Module):
    """Small transformer that processes bytes WITHIN each patch before global model.
    Uses causal attention within patch boundaries (block-causal)."""
    def __init__(self, byte_dim, num_layers, num_heads):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(nn.ModuleDict({
                'attn': Attn(byte_dim, num_heads, num_heads, 1.5),
                'mlp': MLP(byte_dim, 2),
                'as_': nn.ParameterList([nn.Parameter(torch.ones(byte_dim))]),
                'ms': nn.ParameterList([nn.Parameter(torch.ones(byte_dim))]),
            }))

    def forward(self, x, patch_size):
        """x: (B, S, D) byte embeddings. Process within patches."""
        B, S, D = x.shape
        n_patches = S // patch_size
        # Reshape to (B * n_patches, patch_size, D) for independent attention per patch
        x = x.reshape(B * n_patches, patch_size, D)
        for layer in self.layers:
            normed = F.rms_norm(x, (x.size(-1),))
            x = x + layer['as_'][0].to(x.dtype)[None,None,:] * layer['attn'](normed)
            normed = F.rms_norm(x, (x.size(-1),))
            x = x + layer['ms'][0].to(x.dtype)[None,None,:] * layer['mlp'](normed)
        # Reshape back
        return x.reshape(B, S, D)

class Itchy(nn.Module):
    def __init__(self, dim, num_layers, nh, nkv, mlp_mult, patch_size,
                 local_layers=0, local_dim=0, softcap=30.0):
        super().__init__()
        self.dim,self.ps,self.sc=dim,patch_size,softcap
        self.be=nn.Embedding(260,dim if local_dim==0 else local_dim)
        self.local_enc=None
        embed_out_dim = local_dim if local_dim > 0 else dim
        if local_layers > 0 and local_dim > 0:
            self.local_enc = LocalEncoder(local_dim, local_layers, max(1, local_dim//32))
        self.pp=nn.Linear(embed_out_dim*patch_size,dim,bias=False)
        ne=num_layers//2;nd=num_layers-ne
        self.ne,self.nd=ne,nd
        self.sw=nn.Parameter(torch.ones(min(ne,nd),dim))
        self.blocks=nn.ModuleList([Block(dim,nh,nkv,mlp_mult,1.5) for _ in range(num_layers)])
        self.up=nn.Linear(dim,patch_size*260,bias=False)

    def forward(self,ids,tgt):
        B,S=ids.shape;P=self.ps
        x=self.be(ids)  # (B, S, embed_dim)
        if self.local_enc is not None:
            x = self.local_enc(x, P)
        x=x.reshape(B,S//P,P*x.size(-1))
        x=F.rms_norm(self.pp(x),(self.dim,))
        x0=x;skips=[]
        for i in range(self.ne): x=self.blocks[i](x,x0);skips.append(x)
        for i in range(self.nd):
            if skips: x=x+self.sw[i].to(x.dtype)[None,None,:]*skips.pop()
            x=self.blocks[self.ne+i](x,x0)
        x=F.rms_norm(x,(x.size(-1),))
        lo=self.up(x).reshape(-1,260)
        lo=self.sc*torch.tanh(lo/self.sc)
        return F.cross_entropy(lo.float(),tgt.reshape(-1))

n_params=lambda m:sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# HARNESS
# ============================================================

DIM = 256
LAYERS = 8
NH, NKV = 8, 4
MLP_MULT = 3
LR = 3e-4
STEPS = 3000
BATCH_BYTES = 8192

def train_and_val(name, model, patch_size):
    seq_len = (512 // patch_size) * patch_size
    if seq_len == 0: seq_len = patch_size
    model = model.to(device).bfloat16()
    p = n_params(model)
    n_patches = seq_len // patch_size
    print(f'\n{"="*60}')
    print(f'{name}: {p:,} params ({p/1e6:.1f}M) | seq={seq_len} | {n_patches} patches')
    print(f'{"="*60}')
    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.01)
    pos=0;t0=time.time();losses=[]
    for step in range(1,STEPS+1):
        usable=(BATCH_BYTES//seq_len)*seq_len
        if usable==0: usable=seq_len
        end=pos+usable+1
        if end>len(train_data): pos=0;end=usable+1
        chunk=train_data[pos:end];pos=end-1
        x=torch.tensor(chunk[:-1].reshape(-1,seq_len),device=device)
        y=torch.tensor(chunk[1:].reshape(-1,seq_len),device=device)
        opt.zero_grad()
        with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
            loss=model(x,y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step%1000==0:
            avg=np.mean(losses[-1000:])
            print(f'  step {step} | bpb {avg/math.log(2):.4f} | {time.time()-t0:.0f}s')
    train_time=time.time()-t0
    vl=[]
    for i in range(0,min(200*seq_len,len(val_data)-seq_len-1),seq_len):
        chunk=val_data[i:i+seq_len+1]
        x=torch.tensor(chunk[:-1].reshape(1,-1),device=device)
        y=torch.tensor(chunk[1:].reshape(1,-1),device=device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
                vl.append(model(x,y).item())
    val=np.mean(vl);bpb=val/math.log(2)
    print(f'  >>> VAL BPB: {bpb:.4f} | {p/1e6:.1f}M params | {train_time:.0f}s')
    return {'name':name,'params':p,'val_bpb':bpb,'time':train_time,'patch':patch_size}

print('Harness ready')

In [ ]:
# ============================================================
# RUN ALL CONFIGS
# ============================================================

results = []

# --- PATCH SIZE EXTENSIONS ---

# A. Patch 8 (previous best)
torch.manual_seed(42)
results.append(train_and_val('A. patch=8 (prev best)',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,8), 8))

# B. Patch 12
torch.manual_seed(42)
results.append(train_and_val('B. patch=12',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,12), 12))

# C. Patch 16
torch.manual_seed(42)
results.append(train_and_val('C. patch=16',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,16), 16))

# --- LOCAL/GLOBAL SPLIT (research finding #2) ---

# D. Patch 8 + 1-layer local encoder (dim 128)
torch.manual_seed(42)
results.append(train_and_val('D. patch=8 +local(1L,128d)',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,8, local_layers=1, local_dim=128), 8))

# E. Patch 12 + 1-layer local encoder (dim 128)
torch.manual_seed(42)
results.append(train_and_val('E. patch=12 +local(1L,128d)',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,12, local_layers=1, local_dim=128), 12))

# F. Patch 16 + 2-layer local encoder (dim 128)
torch.manual_seed(42)
results.append(train_and_val('F. patch=16 +local(2L,128d)',
    Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,16, local_layers=2, local_dim=128), 16))

# --- LARGER PATCH + LONGER SEQUENCE ---

# G. Patch 8 but seq_len=1024 (2x context)
torch.manual_seed(42)
m = Itchy(DIM,LAYERS,NH,NKV,MLP_MULT,8)
# Override seq_len in harness
m_model = m.to(device).bfloat16()
p = n_params(m_model)
seq_len_g = 1024
print(f'\n{"="*60}')
print(f'G. patch=8 seq=1024: {p:,} params ({p/1e6:.1f}M) | {seq_len_g//8} patches')
print(f'{"="*60}')
opt=torch.optim.AdamW(m_model.parameters(),lr=LR,betas=(0.9,0.95),weight_decay=0.01)
pos=0;t0=time.time();losses_g=[]
for step in range(1,STEPS+1):
    usable=(BATCH_BYTES//seq_len_g)*seq_len_g
    if usable==0: usable=seq_len_g
    end=pos+usable+1
    if end>len(train_data): pos=0;end=usable+1
    chunk=train_data[pos:end];pos=end-1
    x=torch.tensor(chunk[:-1].reshape(-1,seq_len_g),device=device)
    y=torch.tensor(chunk[1:].reshape(-1,seq_len_g),device=device)
    opt.zero_grad()
    with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
        loss=m_model(x,y)
    loss.backward();opt.step()
    losses_g.append(loss.item())
    if step%1000==0:
        avg=np.mean(losses_g[-1000:])
        print(f'  step {step} | bpb {avg/math.log(2):.4f} | {time.time()-t0:.0f}s')
tt=time.time()-t0
vl=[]
for i in range(0,min(200*seq_len_g,len(val_data)-seq_len_g-1),seq_len_g):
    chunk=val_data[i:i+seq_len_g+1]
    x=torch.tensor(chunk[:-1].reshape(1,-1),device=device)
    y=torch.tensor(chunk[1:].reshape(1,-1),device=device)
    with torch.no_grad():
        with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
            vl.append(m_model(x,y).item())
val_g=np.mean(vl)/math.log(2)
print(f'  >>> VAL BPB: {val_g:.4f} | {p/1e6:.1f}M params | {tt:.0f}s')
results.append({'name':'G. patch=8 seq=1024','params':p,'val_bpb':val_g,'time':tt,'patch':8})

In [ ]:
# ============================================================
# RESULTS
# ============================================================

print()
print('='*70)
print('OPTIMAL MODEL SEARCH RESULTS')
print('='*70)
print(f'{"Config":<30} {"Params":>8} {"Val BPB":>9} {"Delta":>9} {"Time":>6}')
print('-'*70)

ref = results[0]['val_bpb']  # patch=8 baseline
best = min(r['val_bpb'] for r in results)
for r in results:
    d = r['val_bpb'] - ref
    tag = ''
    if r['val_bpb'] == ref and r['name'].startswith('A'): tag = ' <-- ref'
    elif r['val_bpb'] == best: tag = ' *** BEST'
    elif d < -0.001: tag = ' +'
    elif d > 0.001: tag = ' -'
    print(f'{r["name"]:<30} {r["params"]/1e6:>7.1f}M {r["val_bpb"]:>9.4f} {d:>+9.4f} {r["time"]:>5.0f}s{tag}')

print()
b = min(results, key=lambda r: r['val_bpb'])
print(f'BEST: {b["name"]} at {b["val_bpb"]:.4f} BPB')
print(f'Previous best (patch=4): 0.5944 BPB')
print(f'Improvement over patch=4: {0.5944 - b["val_bpb"]:+.4f} BPB')